# TCN Training — Temporal Convolutional Network

Trains the **improved TCN** for EEG seizure detection.

### Architecture
| Component | Details |
|-----------|--------|
| Input projection | Conv1d(23→64, k=1) + BN + ReLU |
| Temporal blocks | 8 × dilated causal conv (dilation 1→128) |
| Receptive field | 1021 samples = full 4s window |
| Pooling head | AvgPool + MaxPool → concat → 256-dim |
| Classifier | Linear(256→64) → ReLU → Dropout(0.5) → Linear(64→1) |

### Anti-overfitting improvements
- Dropout = **0.5** (increased from 0.3)
- Weight decay = **5e-4** (stronger L2)
- **Cosine annealing** LR schedule with warm restarts
- **Input noise augmentation** (σ=0.05) during training
- Patience = **10** (more room to find global optimum)

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import warnings; warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt
import json, time
from pathlib import Path

from src.models import TCNClassifier, FocalLoss
from src.training.trainer import run_epoch
from src.data.shared_loaders import get_loaders

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
train_loader, val_loader, test_loader, meta = get_loaders('config.yaml', subsampled=True)
print('DataLoaders ready.')
print(f'  Train : {meta["train"]["n"]:,}  seizure: {meta["train"]["n_seizure"]:,} ({meta["train"]["seizure_frac"]:.2%})')
print(f'  Val   : {meta["val"]["n"]:,}  seizure: {meta["val"]["n_seizure"]:,} ({meta["val"]["seizure_frac"]:.2%})')
print(f'  Test  : {meta["test"]["n"]:,}  seizure: {meta["test"]["n_seizure"]:,} ({meta["test"]["seizure_frac"]:.2%})')

---
## Model Definition & Forward Function

In [ ]:
tcn = TCNClassifier(
    n_channels=23,
    proj_channels=64,
    num_filters=128,
    kernel_size=3,
    num_blocks=8,
    dropout=0.5,
)

params = sum(p.numel() for p in tcn.parameters() if p.requires_grad)
print(f'TCN parameters       : {params:,}')
print(f'Receptive field      : {tcn.receptive_field} samples ({tcn.receptive_field/256:.2f}s at 256Hz)')
print(f'Full window coverage : 1024 samples = 4.00s')

# Forward pass trace
dummy = torch.randn(4, 23, 1024)
tcn.eval()
with torch.no_grad():
    x = tcn.input_proj(dummy)
    print(f'\nForward pass trace (batch=4, C=23, T=1024):')
    print(f'  Input proj       → {tuple(x.shape)}')
    for i, block in enumerate(tcn.network):
        x = block(x)
        print(f'  Block {i} (d={2**i:>3})    → {tuple(x.shape)}')
    avg = tcn.avg_pool(x).squeeze(-1)
    mx  = tcn.max_pool(x).squeeze(-1)
    pool = torch.cat([avg, mx], dim=1)
    print(f'  Dual pool        → {tuple(pool.shape)}')
    out = tcn.classifier(pool)
    print(f'  Classifier       → {tuple(out.shape)}')

---
## Training with Anti-Overfitting Setup

In [ ]:
def add_noise(x, sigma=0.05):
    """Gaussian noise augmentation applied during training."""
    return x + sigma * torch.randn_like(x)


def train_tcn(model, train_loader, val_loader, device,
              lr=3e-4, weight_decay=5e-4, max_epochs=50,
              patience=10, noise_sigma=0.05):

    ckpt_path = CKPT_DIR / 'TCN_best.pt'
    log_path  = CKPT_DIR / 'TCN_log.json'

    model = model.to(device)

    if ckpt_path.exists() and log_path.exists():
        print('[TCN] Loading saved checkpoint.')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        return json.load(open(log_path))

    criterion = FocalLoss(alpha=0.85, gamma=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    # Cosine annealing with warm restarts — escapes local minima
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    history = {'train_loss':[], 'train_f1':[], 'val_loss':[], 'val_f1':[],
               'best_epoch':0, 'lr':[]}
    best_val_f1, no_improve = 0.0, 0

    print(f'[TCN] Training up to {max_epochs} epochs  '
          f'(wd={weight_decay}, noise_sigma={noise_sigma})...')

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()

        # ── Training with noise augmentation ──────────────────────────
        model.train()
        total_loss, all_probs, all_labels = 0.0, [], []
        for X_batch, y_batch in train_loader:
            X_batch = add_noise(X_batch.to(device), sigma=noise_sigma)
            y_batch = y_batch.to(device)
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
            all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

        import numpy as _np
        probs  = _np.array(all_probs)
        labels = _np.array(all_labels)
        preds  = (probs >= 0.5).astype(int)
        tp = int(((preds==1)&(labels==1)).sum())
        fp = int(((preds==1)&(labels==0)).sum())
        fn = int(((preds==0)&(labels==1)).sum())
        sens = tp / max(tp+fn, 1); prec = tp / max(tp+fp, 1)
        tr = {'loss': total_loss/len(labels), 'f1': 2*sens*prec/max(sens+prec, 1e-8)}

        vl = run_epoch(model, val_loader, criterion, None, device, threshold=0.5)
        scheduler.step()

        cur_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(tr['loss']); history['train_f1'].append(tr['f1'])
        history['val_loss'].append(vl['loss']);   history['val_f1'].append(vl['f1'])
        history['lr'].append(cur_lr)

        print(f'  Ep {epoch:02d}/{max_epochs}  '
              f'train loss={tr["loss"]:.4f} f1={tr["f1"]:.4f}  '
              f'val loss={vl["loss"]:.4f} f1={vl["f1"]:.4f}  '
              f'lr={cur_lr:.2e}  ({time.time()-t0:.0f}s)')

        if vl['f1'] > best_val_f1:
            best_val_f1 = vl['f1']; history['best_epoch'] = epoch
            torch.save(model.state_dict(), ckpt_path); no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop (best epoch {history["best_epoch"]})')
                break

    json.dump(history, open(log_path, 'w'), indent=2)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'\nBest Val F1={best_val_f1:.4f} at epoch {history["best_epoch"]} → {ckpt_path.name}')
    return history

In [ ]:
print('TCN Training\n' + '='*55)
hist_tcn = train_tcn(tcn, train_loader, val_loader, DEVICE)

---
## Training Curves

In [ ]:
epochs = range(1, len(hist_tcn['train_loss'])+1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(epochs, hist_tcn['train_loss'], '--', color='#8e44ad', alpha=0.7, label='Train')
axes[0].plot(epochs, hist_tcn['val_loss'],   '-',  color='#8e44ad',            label='Val')
axes[0].axvline(hist_tcn['best_epoch'], color='red', linestyle=':', label=f'Best ep {hist_tcn["best_epoch"]}')
axes[0].set_title('TCN — Focal Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

# F1
axes[1].plot(epochs, hist_tcn['train_f1'], '--', color='#8e44ad', alpha=0.7, label='Train')
axes[1].plot(epochs, hist_tcn['val_f1'],   '-',  color='#8e44ad',            label='Val')
axes[1].axvline(hist_tcn['best_epoch'], color='red', linestyle=':')
axes[1].set_title(f'TCN — F1 (best val={max(hist_tcn["val_f1"]):.4f})', fontweight='bold')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)

# LR schedule
axes[2].plot(epochs, hist_tcn['lr'], color='steelblue')
axes[2].set_title('Learning Rate Schedule (CosineWarmRestarts)', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR'); axes[2].grid(alpha=0.3)

plt.suptitle('TCN Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Report_Figures/tcn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → Report_Figures/tcn_training_curves.png')

In [ ]:
print('TCN Training Summary')
print('='*50)
print(f'  Total epochs  : {len(hist_tcn["train_loss"])}')
print(f'  Best epoch    : {hist_tcn["best_epoch"]}')
print(f'  Best Val F1   : {max(hist_tcn["val_f1"]):.4f}')
print(f'  Final train F1: {hist_tcn["train_f1"][-1]:.4f}')
gap = hist_tcn['train_f1'][hist_tcn['best_epoch']-1] - max(hist_tcn['val_f1'])
print(f'  Train-val gap : {gap:.4f}  (lower = less overfitting)')
print(f'\nCheckpoint saved: checkpoints/TCN_best.pt')